# MindMate - Mental Wellness Chat Prototype
Context-aware chatbot with safety guardrails (PS 03, SDG 3)

**Before running:**
1. Get a free Gemini API key from https://aistudio.google.com
2. In Colab, click the **key icon (Secrets)** on the left, add a secret named `GEMINI_API_KEY`, paste your key, and turn on notebook access.
3. Run the cells one by one from top to bottom.

## Step 1: Install libraries

In [2]:
!pip install -q google-genai gradio transformers sentence-transformers

## Step 2: Connect to Gemini

In [ ]:
from google import genai
from google.genai import types
from google.colab import userdata
userdata.get('GEMINI_API_KEY')

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

# Updated to gemini-3.8-flash because gemini-2.5-flash is deprecated for new users
GEMINI_MODEL = "gemini-3.8-flash"   # NOT a tts/audio model

print(client.models.generate_content(model=GEMINI_MODEL, contents="Say hello in 5 words").text)

In [12]:
for m in client.models.list():
    if "flash" in m.name:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-omni-1.2-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview


## Step 3: Load the Hugging Face models
This takes 1-3 minutes the first time.

In [16]:
from transformers import pipeline
from sentence_transformers import SentenceTransformer, util

# Model 1: emotion detection
emotion_model = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", top_k=None)

# Model 2: sentiment (mood)
sentiment_model = pipeline("text-classification", model="cardiffnlp/twitter-roberta-base-sentiment-latest")

# Model 4: toxic check for the AI reply
toxic_model = pipeline("text-classification", model="unitary/toxic-bert", top_k=None)

# Model 5: exercise matching
match_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Model 3: risk level (big and slow). Set to False if Colab is slow; Gemini will do the risk check instead.
USE_BART = True
if USE_BART:
    risk_model = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

print("All models loaded")

config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  329MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

... (40 outputs left out)

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

All models loaded


## Step 4: Our data (helplines, keywords, exercises, rules)

In [ ]:
HELPLINES = """
### You are not alone. Please reach out now:
- **Tele-MANAS:** 14416 (free, 24/7)
- **KIRAN Mental Health Helpline:** 1800-599-0019
- **Emergency:** 112

Please talk to someone you trust right now. A real person can help you through this.
"""

# Clear crisis phrases (first layer of safety)
CRISIS_KEYWORDS = ["suicide", "suicidal", "kill myself", "end my life", "want to die", "wanna die",
                   "self harm", "self-harm", "hurt myself", "hurting myself", "harm myself", "harming myself",
                   "cut myself", "cutting myself", "no reason to live", "don't want to live", "dont want to live",
                   "better off dead", "end it all", "not worth living"]

EXERCISES = {
    "Box breathing": "Breathe in for 4 seconds, hold for 4, breathe out for 4, hold for 4. Repeat 4 times.",
    "5-4-3-2-1 grounding": "Name 5 things you see, 4 you can touch, 3 you hear, 2 you smell, 1 you taste.",
    "Journal prompt": "Write down one thing that went okay today, even if it was small.",
    "Mindful break": "Close your eyes for 1 minute and just notice your breathing without changing it.",
    "Reach out": "Send a short message to a friend or family member you trust.",
}
# What each exercise is good for (used by the matching model)
EXERCISE_USE = {
    "Box breathing": "panic, anxiety, nervous, fast heartbeat, exam stress, cannot calm down",
    "5-4-3-2-1 grounding": "overthinking, racing thoughts, feeling overwhelmed, cannot focus",
    "Journal prompt": "sad, low mood, bad day, confused feelings",
    "Mindful break": "tired, stressed from work, need a break, cannot sleep",
    "Reach out": "lonely, nobody talks to me, isolated, missing family",
}
exercise_names = list(EXERCISE_USE.keys())
exercise_vectors = match_model.encode(list(EXERCISE_USE.values()), convert_to_tensor=True)

SYSTEM_PROMPT = """You are MindMate, a kind and calm mental wellness companion.
Rules:
- Keep replies short (3 to 4 sentences) and warm.
- Use the user's earlier messages and mood to make the reply personal.
- Suggest simple self-help like breathing, grounding or journaling.
- Never diagnose any illness. Never name or suggest any medicine.
- If asked, say clearly that you are an AI, not a therapist.
- Gently suggest talking to a trusted person or counsellor when the user feels low for long."""


## Step 5: Helper functions (safety checks, emotion, exercise)

In [ ]:
import csv, datetime

def flat(result):
    # Some pipeline versions return [[...]] and some return [...]
    return result[0] if isinstance(result[0], list) else result

def check_risk(text):
    """Layer 1: input safety check. Returns safe / distress / crisis."""
    low = text.lower()
    if any(k in low for k in CRISIS_KEYWORDS):
        return "crisis"
    if USE_BART:
        labels = ["everyday worry or stress about exams, work or life",
                  "feeling hopeless, empty or unable to cope",
                  "wanting to hurt myself or end my life"]
        r = risk_model(text, candidate_labels=labels)
        sc = dict(zip(r["labels"], r["scores"]))
        if sc[labels[2]] > 0.35:      # be extra careful with crisis
            return "crisis"
        if sc[labels[1]] > 0.55:      # only clear hopelessness counts as distress
            return "distress"
        return "safe"
    # Backup: ask Gemini
    q = ("Classify this message for mental health risk. Answer with ONE word: SAFE, DISTRESS or CRISIS. "
         "If unsure between DISTRESS and CRISIS, answer CRISIS.\nMessage: " + text)
    ans = client.models.generate_content(model=GEMINI_MODEL, contents=q).text.strip().upper()
    return "crisis" if "CRISIS" in ans else "distress" if "DISTRESS" in ans else "safe"

def detect_emotions(text, threshold=0.30):
    """Returns all emotions above the threshold (handles mixed emotions)."""
    scores = flat(emotion_model(text[:1000]))
    found = [s["label"] for s in sorted(scores, key=lambda x: -x["score"]) if s["score"] >= threshold]
    return found or ["neutral"]

def detect_sentiment(text):
    return sentiment_model(text[:1000])[0]["label"]

def pick_exercise(text):
    v = match_model.encode(text, convert_to_tensor=True)
    best = int(util.cos_sim(v, exercise_vectors).argmax())
    name = exercise_names[best]
    return name, EXERCISES[name]

BLOCK_WORDS = ["you have depression", "you have anxiety disorder", "diagnos", "mg", "tablet",
               "antidepressant", "prescription", "medication"]

def is_reply_safe(reply):
    """Layer 6: output safety check."""
    tox = {s["label"]: s["score"] for s in flat(toxic_model(reply[:1000]))}
    if tox.get("toxic", 0) > 0.5:
        return False
    if any(w in reply.lower() for w in BLOCK_WORDS):
        return False
    return True

def save_log(mood, emotions, risk):
    with open("mood_log.csv", "a", newline="") as f:
        csv.writer(f).writerow([datetime.datetime.now().strftime("%Y-%m-%d %H:%M"), mood, "+".join(emotions), risk])

# Quick test
for msg in ["I have an exam tomorrow and I'm scared", "I got the internship but I'm nervous", "Nobody talks to me at college"]:
    print(msg, "->", check_risk(msg), detect_emotions(msg), detect_sentiment(msg), pick_exercise(msg)[0])

## Step 6: The full pipeline

In [ ]:
def to_messages(history):
    # Works with both old (tuples) and new (dict) Gradio history formats
    msgs = []
    for h in history:
        if isinstance(h, dict):
            msgs.append((h["role"], h["content"]))
        else:
            if h[0]: msgs.append(("user", h[0]))
            if h[1]: msgs.append(("assistant", h[1]))
    return msgs[-10:]   # keep only the last 10 messages

def mindmate(message, history, mood):
    # Layer 1: input safety check
    risk = check_risk(message)
    if risk == "crisis":
        save_log(mood, ["crisis"], risk)
        return HELPLINES

    # Layer 2: understand the user
    emotions = detect_emotions(message)
    sentiment = detect_sentiment(message)

    # Layer 3: add context
    past = "\n".join(f"{r}: {c}" for r, c in to_messages(history))
    prompt = (f"Earlier conversation:\n{past}\n\n"
              f"User's mood today (1 = very low, 5 = very good): {mood}\n"
              f"Detected emotions: {', '.join(emotions)} | Sentiment: {sentiment}\n"
              f"Risk level: {risk}\n\n"
              f"User says: {message}")
    if risk == "distress":
        prompt += "\n(The user seems very low. Be extra gentle and suggest talking to someone they trust.)"

    # Layer 4: Gemini writes the reply
    reply = client.models.generate_content(
        model=GEMINI_MODEL, contents=prompt,
        config=types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT, temperature=0.4)).text or ""

    # Layer 6: output safety check
    if not is_reply_safe(reply):
        reply = "I hear you, and what you're feeling matters. I'm not able to give medical advice, but talking to a counsellor or doctor can really help."

    # Layer 5: exercise matching
    ex_name, ex_text = pick_exercise(message)

    save_log(mood, emotions, risk)
    return (f"{reply}\n\n**Try this - {ex_name}:** {ex_text}\n\n"
            f"_Detected: {', '.join(emotions)} | risk: {risk}_  \nDid this help? Tell me yes or no.")

In [24]:
import traceback

def safe_mindmate(message, history, mood):
    try:
        return mindmate(message, history, mood)
    except Exception as e:
        traceback.print_exc()
        return f"⚠️ Error: {type(e).__name__}: {e}"

## Step 7: Launch the chat app
Click the **public link** (gradio.live) to open the chat in a new tab.

In [ ]:
import gradio as gr

app = gr.ChatInterface(
    fn=safe_mindmate,
    title="MindMate - Your Wellness Companion",
    description="I'm an AI, not a therapist. In an emergency call Tele-MANAS 14416 or 112.",
    additional_inputs=[gr.Slider(1, 5, value=3, step=1, label="How are you feeling today? (1 = very low, 5 = very good)")],
)
app.launch(share=True, debug=True)

## Step 8 (for Review 3): Test the models
Run this after stopping the app (click the stop button on Step 7).

In [21]:
tests = [
    ("I have an exam tomorrow and I'm scared", "safe"),
    ("Work is so stressful I can't focus", "safe"),
    ("I got the internship but I'm nervous", "safe"),
    ("Nobody talks to me at college", "distress"),
    ("I feel hopeless and can't cope anymore", "distress"),
    ("I want to end my life", "crisis"),
    ("I keep thinking about hurting myself", "crisis"),
]
correct = 0
for msg, expected in tests:
    got = check_risk(msg)
    correct += got == expected
    print(f"{msg[:45]:45} | expected: {expected:8} | got: {got:8} | {'OK' if got == expected else 'MISS'}")
print(f"\nRisk accuracy: {correct}/{len(tests)}")

I have an exam tomorrow and I'm scared        | expected: safe     | got: distress | MISS
Work is so stressful I can't focus            | expected: safe     | got: distress | MISS
I got the internship but I'm nervous          | expected: safe     | got: distress | MISS
Nobody talks to me at college                 | expected: distress | got: distress | OK
I feel hopeless and can't cope anymore        | expected: distress | got: distress | OK
I want to end my life                         | expected: crisis   | got: crisis   | OK
I keep thinking about hurting myself          | expected: crisis   | got: distress | MISS

Risk accuracy: 3/7
